# Week 7 experiment: first 16-base-code multilabel ECG baseline

**Prediction target.** Each output indicates whether an ECG record is linked to one of 16 selected **base ICD-10 code strings**. The immutable source vocabulary is the 19 exact strings in `Data/DiseaseCode.csv`; the mapping is saved in `target_mapping.json`. Multiple codes can be positive on the same record. These are deposited inpatient diagnosis annotations linked to ECGs, not ECG diagnostic statements or independently adjudicated ECG-time disease states. Zero for an unlisted selected code means *not annotated among the selected codes*, **not** clinically confirmed disease absence.

**Data boundary.** Historical patient-level sorted-ID seed-42 70/15/15 split. The first metadata pass indexes only `Patient_ID` to reconstruct membership. The second pass rejects held-out patients **before** reading diagnosis or waveform fields. Train and validation only; no held-out labels or waveforms are accessed. The existing Week 4–6 notebooks, checkpoints, and results remain untouched.

**Experiment.** The Week 5 ResNet-1D architecture is trained from random initialization with 16 logits and unweighted `BCEWithLogitsLoss`. This is a single baseline, not an architecture comparison. Existing binary checkpoints are not loaded. Training uses the same raw first-10-second waveform preparation as the Week 5 disease-labelled experiments: 500 Hz, first 5,000 samples, zero padding if short, 12 anatomical lead slots, absent leads zero, z-score of each present lead over observed samples. Batch 32, Adam learning rate 3e-4 and weight decay 1e-4, at most eight epochs, early stopping patience three, seed 42. The checkpoint minimizes unweighted validation BCE; fixed probability threshold 0.5 is used for descriptive threshold metrics. No validation threshold search or test tuning.

**Evaluation.** Per-target support by records and unique patients, confusion counts, sensitivity, specificity, precision, F1, AUROC and average precision (AP), plus micro and macro summaries. Undefined statistics are `NaN`, never imputed as zero. Very sparse targets are flagged; aggregate metrics state which targets contribute. AP must be interpreted relative to prevalence. The binary 19-code results answer a different task and are not a direct performance comparator.

In [1]:
from pathlib import Path
import csv, random, re, json, sys, os, platform, importlib.util
from collections import Counter
import numpy as np
import pandas as pd
import wfdb, torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, average_precision_score
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'Data/AttributesDictionary.csv').is_file())
OUT=ROOT/'new work/Week7/experimental/multilabel_base16'
assert OUT.is_dir()
assert Path(sys.executable).resolve()==(ROOT/'venv/bin/python').resolve(), 'Select the existing project venv kernel.'
print('Python',sys.executable,platform.python_version(),'PyTorch',torch.__version__,'CUDA build',torch.version.cuda)
assert torch.cuda.is_available(), 'Stop: CUDA is unavailable in this notebook kernel.'
DEVICE=torch.device('cuda:0')
assert (torch.ones(1,device=DEVICE)*2).item()==2.0
print('CUDA verified in kernel:',torch.cuda.get_device_name(0))
print('WFDB',wfdb.__version__,'NumPy',np.__version__,'pandas',pd.__version__)
SEED=42;FS=500;N_SAMPLES=5000;BATCH_SIZE=32;MAX_EPOCHS=8;PATIENCE=3;LR=3e-4;WEIGHT_DECAY=1e-4;THRESHOLD=0.5
LEADS=['I','II','III','AVR','AVL','AVF','V1','V2','V3','V4','V5','V6']
LEAD_INDEX={v:i for i,v in enumerate(LEADS)}
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED);torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True;torch.backends.cudnn.benchmark=False
torch.set_num_threads(min(4,os.cpu_count() or 1))

Python /home/souhit/Documents/Research/Pediatric_ECG/venv/bin/python 3.13.5 PyTorch 2.6.0+cu124 CUDA build 12.4
CUDA verified in kernel: NVIDIA GeForce RTX 3060 Laptop GPU
WFDB 4.3.1 NumPy 2.5.2 pandas 2.3.3


## Original 19 strings and derived 16 targets

The mapping is only a stripping of the two known local qualifiers for `I40.0` and the two known qualifiers for `Q21.1`. Unselected ICD strings in a record are never promoted to positive targets. Target order is fixed by first appearance in the 19-row codebook, and is saved with the checkpoint.

In [2]:
metadata_path=ROOT/'Data/AttributesDictionary.csv'
with metadata_path.open(newline='') as handle:
    reader=csv.reader(handle);header=next(reader);pi=header.index('Patient_ID')
    id_counts=Counter(row[pi] for row in reader)  # Patient_ID only
patient_ids=sorted(id_counts)
random.Random(SEED).shuffle(patient_ids)
n_train=int(.70*len(patient_ids));n_val=int(.15*len(patient_ids))
train_ids=set(patient_ids[:n_train]);val_ids=set(patient_ids[n_train:n_train+n_val]);heldout_ids=set(patient_ids[n_train+n_val:])
assert len(patient_ids)==11643 and (len(train_ids),len(val_ids),len(heldout_ids))==(8150,1746,1747)
assert not (train_ids&val_ids or train_ids&heldout_ids or val_ids&heldout_ids)
book=pd.read_csv(ROOT/'Data/DiseaseCode.csv',keep_default_na=False)
selected_book=book.loc[book['Disease Type']!='Other diseases(OD)'].copy()
exact_codes=selected_book['ICD-10 Code'].str.strip().tolist()
assert len(exact_codes)==len(set(exact_codes))==19
base_of={c:re.sub(r'^\([^)]+\)\s*','',c) for c in exact_codes}
base_codes=list(dict.fromkeys(base_of.values()))
assert len(base_codes)==16
assert base_of['(F) I40.0']==base_of['(V) I40.0']=='I40.0'
assert base_of['Q21.1']==base_of['(FO) Q21.1']==base_of['(OSD) Q21.1']=='Q21.1'
assert all(base_of[c]==c for c in exact_codes if c not in ['(F) I40.0','(V) I40.0','(FO) Q21.1','(OSD) Q21.1'])
base_index={c:i for i,c in enumerate(base_codes)}
exact_set=set(exact_codes)
mapping={'source':'Data/DiseaseCode.csv; 19 selected exact strings excluding Other diseases(OD)',
         'exact_codes':exact_codes,'base_codes_in_model_order':base_codes,
         'exact_to_base':base_of,'rule':'Only selected exact tokens can activate targets; local qualifiers stripped from I40.0 and Q21.1 variants.'}
(OUT/'target_mapping.json').write_text(json.dumps(mapping,indent=2)+'\n')
display(pd.DataFrame({'exact':exact_codes,'base':[base_of[c] for c in exact_codes]}))

,exact,base
0,(F) I40.0,I40.0
1,(V) I40.0,I40.0
2,I40.9,I40.9
3,I51.4,I51.4
4,I42.0,I42.0
5,I42.2,I42.2
6,I42.9,I42.9
7,Q24.8,Q24.8
8,M30.3,M30.3
9,Q21.0,Q21.0


In [3]:
def icd_tokens(value):
    return [x.strip().strip("'\"").strip() for x in value.split(';') if x.strip()]
train_rows=[];val_rows=[]
with metadata_path.open(newline='') as handle:
    for row in csv.DictReader(handle):
        pid=row['Patient_ID']
        if pid in heldout_ids:
            continue  # No held-out diagnosis, age, lead, sampling or waveform access.
        assert pid in train_ids or pid in val_ids
        assert row['Filename'].split('/')[-2]==pid
        exact=set(icd_tokens(row['ICD-10 code']))&exact_set
        bases={base_of[c] for c in exact}
        target=np.zeros(16,dtype=np.float32)
        for code in bases:target[base_index[code]]=1.0
        assert re.fullmatch(r'\d+d',row['Age'])
        item={'patient_id':pid,'ecg_id':row['ECG_ID'],'filename':row['Filename'],
              'sampling_point':int(row['Sampling_point']),'lead_count':int(row['Lead']),
              'age_years':int(row['Age'][:-1])/365.25,'exact_codes':sorted(exact),'target':target}
        (train_rows if pid in train_ids else val_rows).append(item)
assert (len(train_rows),len(val_rows))==(9943,2169)
assert (sum(bool(r['exact_codes']) for r in train_rows),sum(bool(r['exact_codes']) for r in val_rows))==(2595,583)
assert all({base_of[c] for c in r['exact_codes']}=={base_codes[i] for i,v in enumerate(r['target']) if v} for r in train_rows+val_rows)
Y_train=np.stack([r['target'] for r in train_rows]);Y_val=np.stack([r['target'] for r in val_rows])
assert Y_train.shape==(9943,16) and Y_val.shape==(2169,16)
assert np.array_equal(Y_train.max(1),[bool(r['exact_codes']) for r in train_rows])
assert np.array_equal(Y_val.max(1),[bool(r['exact_codes']) for r in val_rows])
assert tuple(Y_train.sum(0)[base_index[c]] for c in ['I40.0','Q21.1'])==(296.0,1158.0)
assert tuple(Y_val.sum(0)[base_index[c]] for c in ['I40.0','Q21.1'])==(57.0,226.0)
support=[]
for split,rows,Y,ids in [('train',train_rows,Y_train,train_ids),('validation',val_rows,Y_val,val_ids)]:
    for j,code in enumerate(base_codes):
        mask=Y[:,j]==1
        support.append({'split':split,'code':code,'records':int(mask.sum()),
                        'patients':len({r['patient_id'] for r,yes in zip(rows,mask) if yes}),
                        'record_prevalence':float(mask.mean()),
                        'patient_prevalence':len({r['patient_id'] for r,yes in zip(rows,mask) if yes})/len(ids)})
support=pd.DataFrame(support)
support.to_csv(OUT/'target_support.csv',index=False)
display(support.pivot(index='code',columns='split',values=['records','patients']).reindex(base_codes).astype(int))
print('Records with >=2 base targets:',int((Y_train.sum(1)>=2).sum()),'train;',int((Y_val.sum(1)>=2).sum()),'validation')

records            patients           
split   train validation    train validation
code                                        
I40.0     296         57      116         27
I40.9      39         21       17          6
I51.4     106         35       82         22
I42.0      44         12       26          8
I42.2      18          6       13          2
I42.9      17          0       11          0
Q24.8      14          6       10          3
M30.3     134         25      116         23
Q21.0     723        196      525        127
Q21.1    1158        226      845        167
Q21.2      35         10       21          8
Q21.3      72         10       38          9
Q22.1      17          1       15          1
Q25.0     236         58      182         43
Q25.6      14          0       11          0
I37.0      62          8       51          6

Records with >=2 base targets: 358 train; 79 validation


## Waveform pipeline and architecture

The old `NoteBooks/Week6/dataset.py` derives labels from ECG statements and is **not** used. The dataset class below adapts the Week 5 disease-labelled loader, retaining lead mapping and normalization. The Week 5 `ResNet1D` architecture is imported without changing its source; only `num_classes=16` differs.

In [4]:
RECORDS=ROOT/'Child_ECG/Child_ecg'
class Base16Dataset(Dataset):
    def __init__(self,rows):self.rows=rows
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        r=self.rows[i]
        signal,fields=wfdb.rdsamp(str(RECORDS/r['filename']),sampto=min(N_SAMPLES,r['sampling_point']))
        assert fields['fs']==FS and np.isfinite(signal).all()
        x=np.zeros((12,N_SAMPLES),dtype=np.float32)
        for col,lead in enumerate(fields['sig_name']):
            key=lead.upper();assert key in LEAD_INDEX
            v=signal[:,col].astype(np.float32);sd=v.std()
            x[LEAD_INDEX[key],:len(v)]=(v-v.mean())/sd if sd>1e-8 else 0
        return torch.from_numpy(x),torch.from_numpy(r['target'])
train_data=Base16Dataset(train_rows);val_data=Base16Dataset(val_rows)
train_loader=DataLoader(train_data,batch_size=BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=True)
val_loader=DataLoader(val_data,batch_size=BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=True)
spec=importlib.util.spec_from_file_location('week5_resnet_arch',ROOT/'NoteBooks/Week5/models.py')
module=importlib.util.module_from_spec(spec);spec.loader.exec_module(module)
ResNet1D=module.ResNet1D
model=ResNet1D(num_classes=16).to(DEVICE)
with torch.no_grad():assert model(torch.zeros(1,12,N_SAMPLES,device=DEVICE)).shape==(1,16)
print('ResNet1D trainable parameters:',sum(p.numel() for p in model.parameters()))
config={'architecture':'NoteBooks/Week5/models.py:ResNet1D(num_classes=16); random initialization',
        'target_mapping':'target_mapping.json','seed':SEED,'patient_split':'sorted IDs, random.Random(42), floor 70/15/15',
        'train_patients':len(train_ids),'validation_patients':len(val_ids),'train_records':len(train_rows),'validation_records':len(val_rows),
        'preprocessing':'raw first 5000 samples at 500 Hz; anatomical 12 lead slots; per-present-lead z-score; zero pad short/missing',
        'lead_order':LEADS,'batch_size':BATCH_SIZE,'max_epochs':MAX_EPOCHS,'early_stopping_patience':PATIENCE,
        'optimizer':'Adam','learning_rate':LR,'weight_decay':WEIGHT_DECAY,
        'loss':'unweighted BCEWithLogitsLoss','checkpoint_selection':'minimum validation BCE, min delta 1e-5',
        'decision_threshold':THRESHOLD,'test_labels_or_waveforms_accessed':False,
        'python':sys.version,'torch':torch.__version__,'cuda_build':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),
        'wfdb':wfdb.__version__,'numpy':np.__version__,'pandas':pd.__version__}
(OUT/'config.json').write_text(json.dumps(config,indent=2)+'\n')

ResNet1D trainable parameters: 550736


1165

## Train once, select by validation BCE

The first baseline uses no class weighting. This intentionally exposes any failure to detect rare targets; it is not evidence that weighting would fail. Epoch history is saved after every epoch. A binary checkpoint is never loaded.

In [5]:
criterion=torch.nn.BCEWithLogitsLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
def run_epoch(loader,training):
    model.train(training);total=0.;out_y=[];out_p=[]
    for x,y in loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        with torch.set_grad_enabled(training):
            logits=model(x);loss=criterion(logits,y)
            if training:
                optimizer.zero_grad(set_to_none=True);loss.backward();optimizer.step()
        total+=loss.item()*len(y)
        if not training:
            out_y.append(y.cpu().numpy());out_p.append(torch.sigmoid(logits).cpu().numpy())
    return total/len(loader.dataset),(np.concatenate(out_y) if out_y else None),(np.concatenate(out_p) if out_p else None)
history=[];best_loss=float('inf');best_epoch=None;stale=0
checkpoint=OUT/'resnet1d_base16_best.pth'
for epoch in range(1,MAX_EPOCHS+1):
    train_loss,_,_=run_epoch(train_loader,True)
    val_loss,y_epoch,p_epoch=run_epoch(val_loader,False)
    assert np.array_equal(y_epoch,Y_val)
    history.append({'epoch':epoch,'train_bce':train_loss,'validation_bce':val_loss})
    pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
    if val_loss<best_loss-1e-5:
        best_loss=val_loss;best_epoch=epoch;stale=0
        torch.save({'model_state_dict':model.state_dict(),'architecture':'ResNet1D','num_classes':16,
                    'base_codes_in_model_order':base_codes,'exact_to_base':base_of,'original_exact_codes':exact_codes,
                    'epoch':epoch,'validation_bce':val_loss,'seed':SEED,'input_samples':N_SAMPLES,'lead_order':LEADS,
                    'selection':'minimum unweighted validation BCE'},checkpoint)
    else:stale+=1
    print(f'Epoch {epoch}: train BCE {train_loss:.5f}, validation BCE {val_loss:.5f}, best epoch {best_epoch}',flush=True)
    if stale>=PATIENCE:break
print('Selected epoch:',best_epoch,'validation BCE:',best_loss)

Epoch 1: train BCE 0.10861, validation BCE 0.07925, best epoch 1


Epoch 2: train BCE 0.06652, validation BCE 0.06396, best epoch 2


Epoch 3: train BCE 0.06393, validation BCE 0.06639, best epoch 2


Epoch 4: train BCE 0.06228, validation BCE 0.06341, best epoch 4


Epoch 5: train BCE 0.06124, validation BCE 0.06229, best epoch 5


Epoch 6: train BCE 0.05982, validation BCE 0.06311, best epoch 5


Epoch 7: train BCE 0.05888, validation BCE 0.06150, best epoch 7


Epoch 8: train BCE 0.05785, validation BCE 0.06112, best epoch 8


Selected epoch: 8 validation BCE: 0.06111971202657252


## Fixed-checkpoint validation

The tables below are validation observations from the retained checkpoint. Threshold metrics use 0.5. AUROC and AP are undefined if the validation target has no positive or no negative records. Sensitivity and F1 are undefined if there are no positives. A target with fewer than ten positive validation patients is marked as sparse, even if a numeric metric exists.

In [6]:
saved=torch.load(checkpoint,map_location=DEVICE,weights_only=False)
assert saved['base_codes_in_model_order']==base_codes and saved['exact_to_base']==base_of
model.load_state_dict(saved['model_state_dict'])
val_loss,y,p=run_epoch(val_loader,False)
assert np.array_equal(y,Y_val) and abs(val_loss-saved['validation_bce'])<1e-6
pred=(p>=THRESHOLD).astype(np.int8)
per_target=[]
for j,code in enumerate(base_codes):
    truth=y[:,j].astype(int);guess=pred[:,j].astype(int);score=p[:,j]
    tp=int(((truth==1)&(guess==1)).sum());tn=int(((truth==0)&(guess==0)).sum())
    fp=int(((truth==0)&(guess==1)).sum());fn=int(((truth==1)&(guess==0)).sum())
    pos=tp+fn;neg=tn+fp
    pos_patients=support.query("split=='validation' and code==@code").patients.iloc[0]
    per_target.append({'code':code,'positive_records':pos,'positive_patients':int(pos_patients),'prevalence':pos/len(truth),
                       'tn':tn,'fp':fp,'fn':fn,'tp':tp,
                       'sensitivity':tp/pos if pos else np.nan,'specificity':tn/neg if neg else np.nan,
                       'precision':tp/(tp+fp) if tp+fp else np.nan,
                       'f1':2*tp/(2*tp+fp+fn) if pos else np.nan,
                       'auroc':roc_auc_score(truth,score) if pos and neg else np.nan,
                       'average_precision':average_precision_score(truth,score) if pos and neg else np.nan,
                       'status':'no_validation_positive' if pos==0 else ('<10_positive_patients' if pos_patients<10 else 'measurable')})
per_target=pd.DataFrame(per_target)
per_target.to_csv(OUT/'per_target_validation.csv',index=False)
display(per_target.round(4))

,code,positive_records,positive_patients,prevalence,tn,fp,fn,tp,sensitivity,specificity,precision,f1,auroc,average_precision,status
0,I40.0,57,27,0.0263,2091,21,43,14,0.2456,0.9901,0.4000,0.3043,0.7143,0.2022,measurable
1,I40.9,21,6,0.0097,2148,0,21,0,0.0000,1.0000,NaN,0.0000,0.7413,0.0247,<10_positive_patients
2,I51.4,35,22,0.0161,2134,0,35,0,0.0000,1.0000,NaN,0.0000,0.5600,0.0180,measurable
3,I42.0,12,8,0.0055,2157,0,11,1,0.0833,1.0000,1.0000,0.1538,0.8828,0.2697,<10_positive_patients
4,I42.2,6,2,0.0028,2163,0,6,0,0.0000,1.0000,NaN,0.0000,0.9021,0.0228,<10_positive_patients
5,I42.9,0,0,0.0000,2169,0,0,0,NaN,1.0000,NaN,NaN,NaN,NaN,no_validation_positive
6,Q24.8,6,3,0.0028,2163,0,6,0,0.0000,1.0000,NaN,0.0000,0.8056,0.0079,<10_positive_patients
7,M30.3,25,23,0.0115,2144,0,25,0,0.0000,1.0000,NaN,0.0000,0.7792,0.0495,measurable
8,Q21.0,196,127,0.0904,1952,21,131,65,0.3316,0.9894,0.7558,0.4610,0.9111,0.6239,measurable
9,Q21.1,226,167,0.1042,1918,25,160,66,0.2920,0.9871,0.7253,0.4164,0.8373,0.5437,measurable


In [7]:
flat_y=y.ravel().astype(int);flat_pred=pred.ravel().astype(int);flat_p=p.ravel()
micro_tp=int(((flat_y==1)&(flat_pred==1)).sum());micro_tn=int(((flat_y==0)&(flat_pred==0)).sum())
micro_fp=int(((flat_y==0)&(flat_pred==1)).sum());micro_fn=int(((flat_y==1)&(flat_pred==0)).sum())
macro_defined=per_target[per_target.positive_records>0]
summary={'checkpoint':checkpoint.name,'selected_epoch':int(saved['epoch']),'validation_bce':float(val_loss),
         'validation_records':len(y),'validation_patients':len(val_ids),'targets_total':len(base_codes),
         'macro_targets_with_positives':len(macro_defined),
         'macro_targets_with_at_least_10_positive_patients':int((per_target.positive_patients>=10).sum()),
         'micro_prevalence':float(flat_y.mean()),'micro_tn':micro_tn,'micro_fp':micro_fp,'micro_fn':micro_fn,'micro_tp':micro_tp,
         'micro_sensitivity':micro_tp/(micro_tp+micro_fn),'micro_specificity':micro_tn/(micro_tn+micro_fp),
         'micro_precision':micro_tp/(micro_tp+micro_fp) if micro_tp+micro_fp else None,
         'micro_f1':2*micro_tp/(2*micro_tp+micro_fp+micro_fn),
         'micro_auroc':float(roc_auc_score(flat_y,flat_p)),'micro_average_precision':float(average_precision_score(flat_y,flat_p)),
         'macro_auroc_defined_targets':float(macro_defined.auroc.mean()),
         'macro_average_precision_defined_targets':float(macro_defined.average_precision.mean()),
         'macro_f1_defined_targets':float(macro_defined.f1.mean()),
         'macro_sensitivity_defined_targets':float(macro_defined.sensitivity.mean()),
         'macro_specificity_defined_targets':float(macro_defined.specificity.mean()),
         'subset_accuracy':float(np.mean(np.all(pred==y,axis=1))),
         'hamming_loss':float(np.mean(pred!=y)),
         'threshold':THRESHOLD,'test_evaluated':False}
(OUT/'aggregate_validation.json').write_text(json.dumps(summary,indent=2)+'\n')
pred_rows=[]
for i,r in enumerate(val_rows):
    d={'patient_id':r['patient_id'],'ecg_id':r['ecg_id'],'age_years':r['age_years'],'lead_count':r['lead_count']}
    for j,code in enumerate(base_codes):
        d[f'target::{code}']=int(y[i,j]);d[f'probability::{code}']=float(p[i,j])
    pred_rows.append(d)
pd.DataFrame(pred_rows).to_csv(OUT/'validation_predictions.csv',index=False)
display(pd.DataFrame([summary]).T.rename(columns={0:'value'}))

,value
checkpoint,resnet1d_base16_best.pth
selected_epoch,8
validation_bce,0.06112
validation_records,2169
validation_patients,1746
targets_total,16
macro_targets_with_positives,14
macro_targets_with_at_least_10_positive_patients,6
micro_prevalence,0.019335
micro_tn,33966


## Observed baseline behaviour

The retained checkpoint is **epoch 8** (lowest validation BCE **0.06112**). On 2,169 validation records from 1,746 patients, the fixed 0.5-threshold micro confusion counts across all 16 outputs are **TN 33,966 / FP 67 / FN 524 / TP 147**. Micro sensitivity is **0.219**, specificity **0.998**, F1 **0.332**, AUROC **0.913**, and AP **0.400**. Macro AP is **0.164** and macro AUROC **0.828**, each averaged over the **14** targets with validation positives; two targets have no validation positives. Only **six** targets have at least ten positive validation patients.

**Observation:** just five targets have any true positives at threshold 0.5; 145 of 147 true positives come from `I40.0`, `Q21.0`, and `Q21.1`. Eleven targets receive no positive predictions at that threshold. `I42.9` and `Q25.6` have zero positive validation records. `Q22.1` has one positive validation patient, so its numerical ranking metrics are not reliable evidence of performance. The apparent high micro AUROC and specificity coexist with missed annotated diagnoses; subset accuracy and Hamming loss are dominated by negative target entries.

**Interpretation:** this run establishes a reproducible first code-presence baseline and shows limited detection at the fixed threshold. Its threshold behaviour could reflect severe class imbalance, score calibration, signal limitations, annotation ambiguity, or their combination. This single run cannot determine which explanation dominates. No threshold or class-weight adjustment was selected on validation in this baseline.

## Interpretation constraints

This run measures prediction of **deposited selected code presence** on an existing validation cohort. The dataset is a hospitalized pediatric cohort; the annotations derive from inpatient records linked to ECGs. Prior Week 6 audit found every train/validation record had an ICD field, yet repeated ECGs for a patient carried identical ICD strings despite differing dates, so temporal disease activity is unresolved. A code absent from the selected list is not a verified absence of that disease. Zero-positive validation targets cannot have sensitivity, AP or AUROC estimated here; metrics for one or a few positive patients are highly unstable. The existing patient-level split is not stratified per code. This notebook does not test clinical utility, generalization, or causal ECG findings.